# LeetCode 2582 — Pass the Pillow bằng Python

Link đề: https://leetcode.com/problems/pass-the-pillow/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def passThePillow(self, n: int, time: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chiếc gối được chuyền đi **qua lại** dọc một hàng người `1..n`, mỗi giây chuyền một bước, tới đầu mút thì **đổi chiều**.
2. Viết được lời giải **mô phỏng** (simulation) bằng biến vị trí `pos` và biến chiều `step` (`+1` hoặc `-1`).
3. Nhận ra chuyển động là **tuần hoàn**: sau `2(n - 1)` giây gối trở lại người 1 với cùng chiều; dùng phép chia lấy dư `%` và `divmod`.
4. Viết 4 cách: mô phỏng, đếm số "lượt đi / lượt về" bằng `divmod`, tra bảng chu kỳ, và công thức "hình tam giác" bằng `abs`.
5. Kiểm chứng bằng test ngẫu nhiên có seed với một oracle xây hẳn dãy người giữ gối.
6. Tránh các lỗi quen thuộc: quên đổi chiều (`time % n + 1`), chu kỳ sai (`2n` thay vì `2(n - 1)`), chia cho `n` thay vì `n - 1`, đổi chiều ngay lúc xuất phát.

## 2. Đề bài & ví dụ

Có `n` người đứng thành một hàng, đánh số từ `1` đến `n`. Người đầu hàng (người 1) đang cầm một chiếc gối.

Mỗi giây, người đang cầm gối **chuyền** nó cho người kế tiếp trong hàng. Khi gối tới **cuối hàng**, chiều chuyền **đảo ngược**, và mọi người tiếp tục chuyền theo chiều ngược lại. Ví dụ: khi gối tới người `n`, họ chuyền cho người `n - 1`, rồi người `n - 2`, ...; tới người 1 thì lại đổi chiều.

Cho hai số nguyên dương `n` và `time`, trả về **số thứ tự** của người đang cầm gối sau `time` giây.

| Ví dụ | n | time | Output | Giải thích |
|---|---|---|---|---|
| 1 | 4 | 5 | 2 | gối đi: `1 → 2 → 3 → 4 → 3 → 2`; sau 5 giây ở người 2 |
| 2 | 3 | 2 | 3 | gối đi: `1 → 2 → 3`; sau 2 giây ở người 3 |

Constraints: `2 <= n <= 1000`; `1 <= time <= 1000`.

In [1]:
n, time = 4, 5
holders = [1, 2, 3, 4, 3, 2]          # ví dụ 1 viết tay theo đề: 1 -> 2 -> 3 -> 4 -> 3 -> 2
for sec, person in enumerate(holders):
    print(f"giây {sec}: người {person}")
print("sau", time, "giây ở người", holders[time], "(đáp án mong đợi: 2)")

giây 0: người 1
giây 1: người 2
giây 2: người 3
giây 3: người 4
giây 4: người 3
giây 5: người 2
sau 5 giây ở người 2 (đáp án mong đợi: 2)


## 3. Kiến thức nền

### 3.1. Biến chiều `step` = `+1` / `-1`

Thay vì viết hai đoạn code (đi tới, đi lui), ta giữ biến `step` cho biết **cộng bao nhiêu** vào vị trí mỗi giây: `step = 1` là đi tới, `step = -1` là đi lui. Đổi chiều chỉ là `step = -step`.

In [2]:
pos, step = 1, 1
for sec in range(1, 4):
    pos += step
    print("giây", sec, "-> vị trí", pos)
step = -step                    # đảo chiều
print("sau khi đảo chiều, step =", step)
pos += step
print("vị trí mới:", pos)

giây 1 -> vị trí 2
giây 2 -> vị trí 3
giây 3 -> vị trí 4
sau khi đảo chiều, step = -1
vị trí mới: 3


### 3.2. Phép chia lấy dư `%` và `divmod`

- `a % b` là phần dư khi chia `a` cho `b`; `a // b` là thương nguyên.
- `divmod(a, b)` trả luôn cặp `(a // b, a % b)`.
- `%` biến một số lớn bất kỳ thành giá trị trong `0..b-1` — hợp cho các chuyển động **lặp lại theo chu kỳ**.

In [3]:
print(17 % 6, 17 // 6, divmod(17, 6))   # 5 2 (2, 5)
for t in range(0, 13):
    print(t, "->", t % 6, end=" | ")

5 2 (2, 5)
0 -> 0 | 1 -> 1 | 2 -> 2 | 3 -> 3 | 4 -> 4 | 5 -> 5 | 6 -> 0 | 7 -> 1 | 8 -> 2 | 9 -> 3 | 10 -> 4 | 11 -> 5 | 12 -> 0 | 

### 3.3. Tính chẵn lẻ của thương

`q % 2 == 0` kiểm tra `q` chẵn. Ta sẽ dùng nó để biết gối đang ở "lượt đi" (thương chẵn) hay "lượt về" (thương lẻ).

In [4]:
for q in range(5):
    print(q, "chẵn" if q % 2 == 0 else "lẻ")

0 chẵn
1 lẻ
2 chẵn
3 lẻ
4 chẵn


### 3.4. Danh sách chu kỳ và chỉ số

Có thể liệt kê **một chu kỳ** các vị trí vào list rồi tra `path[time % len(path)]`. Hai hàm hay dùng: `range(a, b)` (tăng) và `range(a, b, -1)` (giảm, dừng **trước** `b`), và phép nối list bằng `+`.

In [5]:
n = 4
up = list(range(1, n + 1))          # [1, 2, 3, 4]
down = list(range(n - 1, 1, -1))    # [3, 2]  (không lặp lại 4 và 1)
print(up, down, up + down)

[1, 2, 3, 4] [3, 2] [1, 2, 3, 4, 3, 2]


### 3.5. Giá trị tuyệt đối `abs`

`abs(x)` bỏ dấu của `x`. Đồ thị `abs(x - c)` có hình chữ V; lấy `h - abs(x - c)` cho hình chữ Λ ("tam giác") — rất hợp để biểu diễn chuyển động đi lên rồi đi xuống.

In [6]:
c = 3
print([abs(x - c) for x in range(7)])        # [3, 2, 1, 0, 1, 2, 3]
print([c - abs(x - c) for x in range(7)])    # [0, 1, 2, 3, 2, 1, 0]

[3, 2, 1, 0, 1, 2, 3]
[0, 1, 2, 3, 2, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cứ làm đúng như đề: viết ra người giữ gối ở từng giây.

`n = 4`:

```
giây:   0 1 2 3 4 5 6 7 8 9 10 11 12
người:  1 2 3 4 3 2 1 2 3 4 3  2  1
```

Quan sát: gối đi `1 → 4` mất `n - 1 = 3` giây, về `4 → 1` mất thêm 3 giây. Cứ `2(n - 1) = 6` giây thì lặp lại hệt (`giây 0` và `giây 6` đều ở người 1, `giây 1` và `giây 7` đều ở người 2...).

In [7]:
n = 4
seq = []
pos, step = 1, 1
for sec in range(13):
    seq.append(pos)
    if pos == n:
        step = -1
    elif pos == 1:
        step = 1
    pos += step
print("giây  :", list(range(13)))
print("người :", seq)

giây  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
người : [1, 2, 3, 4, 3, 2, 1, 2, 3, 4, 3, 2, 1]


## 5. Cách 1 — Brute Force / baseline

**Mô phỏng từng giây**: giữ `pos` (đang ở ai) và `step` (chiều). Trước mỗi lần chuyền, nếu `pos` đang ở đầu mút thì đảo chiều cho đúng, rồi chuyền.

- Đang ở người `n` → phải đi **lùi** (`step = -1`).
- Đang ở người `1` → phải đi **tiến** (`step = 1`).

In [8]:
def pillow_simulate(n: int, time: int) -> int:
    pos, step = 1, 1
    for _ in range(time):
        if pos == n:
            step = -1
        elif pos == 1:
            step = 1
        pos += step
    return pos

In [9]:
print(pillow_simulate(4, 5))    # 2
print(pillow_simulate(3, 2))    # 3
print(pillow_simulate(2, 1))    # 2

2
3
2


**Độ phức tạp:** vòng lặp chạy `time` lần, mỗi lần `O(1)` → Time `O(time)`; Space `O(1)`.

**Nút thắt:** nếu `time` rất lớn (ví dụ `10^18`) thì mô phỏng không kịp. Đề chỉ cho `time <= 1000` nên vẫn qua, nhưng ta có thể **tính thẳng** nhờ tính tuần hoàn.

## 6. Ý tưởng thuật toán / bước đột phá

Gối đi hết một "lượt" (từ đầu này tới đầu kia) mất **đúng `n - 1` giây** (qua `n - 1` khoảng giữa các người).

- Chia `time` cho `n - 1`: `q, r = divmod(time, n - 1)`. `q` là **số lượt đã đi xong**, `r` là số bước đã đi trong lượt hiện tại.
- Lượt chẵn (`q` chẵn: lượt 0, 2, ...) gối đi **tiến**, bắt đầu từ người `1` → sau `r` bước ở người `1 + r`.
- Lượt lẻ (`q` lẻ) gối đi **lùi**, bắt đầu từ người `n` → sau `r` bước ở người `n - r`.

Quy tắc "nếu… thì…": `q` chẵn → `1 + r`; `q` lẻ → `n - r`.

In [10]:
n, time = 4, 5
q, r = divmod(time, n - 1)
print("q =", q, ", r =", r)
print("người giữ gối:", 1 + r if q % 2 == 0 else n - r)     # q = 1 (lẻ) -> n - r = 4 - 2 = 2

q = 1 , r = 2
người giữ gối: 2


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** `divmod` theo từng lượt (công thức ở mục 6), **(3)** tra bảng chu kỳ, **(4)** công thức `abs` hình tam giác.

### Cách 2 — `divmod(time, n - 1)` rồi xét chẵn / lẻ

Giải thích từng đoạn:

1. `q, r = divmod(time, n - 1)` — `q` lượt đầy đủ, `r` bước lẻ.
2. Nếu `q` chẵn thì gối đang trên "đường đi" (tiến): người `1 + r`.
3. Nếu `q` lẻ thì gối đang trên "đường về": người `n - r`.

In [11]:
def pillow_divmod(n, time):
    q, r = divmod(time, n - 1)
    if q % 2 == 0:
        return 1 + r
    return n - r

print(pillow_divmod(4, 5), pillow_divmod(3, 2))

2 3


In [12]:
class SolutionDivmod:
    def passThePillow(self, n: int, time: int) -> int:
        q, r = divmod(time, n - 1)
        return 1 + r if q % 2 == 0 else n - r

Độ phức tạp Cách 2: Time `O(1)`; Space `O(1)`.

### Cách 3 — Bảng một chu kỳ

Một chu kỳ dài `2(n - 1)` giây. Dựng list `path = [1, 2, ..., n, n-1, ..., 2]` (không lặp lại người 1 ở cuối, vì đó là giây đầu của chu kỳ sau). Đáp án là `path[time % len(path)]`.

Với `n = 4`: `path = [1, 2, 3, 4, 3, 2]` có 6 phần tử = `2(n - 1)`.

In [13]:
def pillow_table(n, time):
    path = list(range(1, n + 1)) + list(range(n - 1, 1, -1))
    return path[time % len(path)]

print(pillow_table(4, 5), pillow_table(3, 2), pillow_table(2, 1))
print([pillow_table(4, t) for t in range(13)])     # trùng với dòng "người" ở mục 4

2 3 2
[1, 2, 3, 4, 3, 2, 1, 2, 3, 4, 3, 2, 1]


In [14]:
class SolutionTable:
    def passThePillow(self, n: int, time: int) -> int:
        path = list(range(1, n + 1)) + list(range(n - 1, 1, -1))
        return path[time % len(path)]

Độ phức tạp Cách 3: Time `O(n)` để dựng bảng; Space `O(n)`. Dễ hiểu nhất để kiểm chứng, nhưng tốn bộ nhớ.

### Cách 4 — Công thức hình tam giác bằng `abs`

Đặt `r = time % (2 * (n - 1))` (vị trí trong một chu kỳ, `0 <= r < 2(n - 1)`). Đồ thị người giữ gối theo `r` là một "tam giác": tăng từ `1` tới `n` tại `r = n - 1`, rồi giảm. Hình tam giác đó là `n - abs(r - (n - 1))`:

- `r = 0` → `n - (n - 1) = 1`;
- `r = n - 1` → `n - 0 = n` (đỉnh);
- `r = 2(n - 1) - 1` → `n - (n - 2) = 2` (người ngay trước khi quay về 1).

In [15]:
def pillow_triangle(n, time):
    r = time % (2 * (n - 1))
    return n - abs(r - (n - 1))

print(pillow_triangle(4, 5), pillow_triangle(3, 2), pillow_triangle(2, 1))
print([pillow_triangle(4, t) for t in range(13)])

2 3 2
[1, 2, 3, 4, 3, 2, 1, 2, 3, 4, 3, 2, 1]


In [16]:
class SolutionTriangle:
    def passThePillow(self, n: int, time: int) -> int:
        r = time % (2 * (n - 1))
        return n - abs(r - (n - 1))

Độ phức tạp Cách 4: Time `O(1)`; Space `O(1)`. Ngắn nhất, nhưng cần hiểu vì sao đúng (mục 10).

## 8. TRACE từng bước

Trace Cách 1 (mô phỏng) với `n = 4`, `time = 5`.

| giây | `pos` trước | `step` | `pos` sau |
|---|---|---|---|
| 1 | 1 | `+1` (ở người 1 → tiến) | 2 |
| 2 | 2 | `+1` | 3 |
| 3 | 3 | `+1` | 4 |
| 4 | 4 | `-1` (ở người n → lùi) | 3 |
| 5 | 3 | `-1` | 2 |

Kết quả `pos = 2`.

In [17]:
def trace_simulate(n, time):
    pos, step = 1, 1
    print(f"bắt đầu: pos={pos}")
    for sec in range(1, time + 1):
        before = pos
        if pos == n:
            step = -1
        elif pos == 1:
            step = 1
        pos += step
        print(f"giây {sec}: pos {before} -> {pos} (step={step:+d})")
    return pos

trace_simulate(4, 5)
trace_simulate(3, 2)

bắt đầu: pos=1
giây 1: pos 1 -> 2 (step=+1)
giây 2: pos 2 -> 3 (step=+1)
giây 3: pos 3 -> 4 (step=+1)
giây 4: pos 4 -> 3 (step=-1)
giây 5: pos 3 -> 2 (step=-1)
bắt đầu: pos=1
giây 1: pos 1 -> 2 (step=+1)
giây 2: pos 2 -> 3 (step=+1)


3

In [18]:
def trace_divmod(n, time):
    q, r = divmod(time, n - 1)
    print(f"n={n}, time={time}: q={q}, r={r}, q {'chẵn' if q % 2 == 0 else 'lẻ'}")
    ans = 1 + r if q % 2 == 0 else n - r
    print("người giữ gối =", ans)
    return ans

trace_divmod(4, 5)
trace_divmod(3, 2)
trace_divmod(4, 6)      # đúng một chu kỳ: về người 1

n=4, time=5: q=1, r=2, q lẻ
người giữ gối = 2
n=3, time=2: q=1, r=0, q lẻ
người giữ gối = 3
n=4, time=6: q=2, r=0, q chẵn
người giữ gối = 1


1

## 9. Code nộp LeetCode

Nộp Cách 2 (`divmod`): ngắn, `O(1)`, vẫn dễ hiểu và không cần nhớ công thức `abs`.

In [19]:
class Solution:
    def passThePillow(self, n: int, time: int) -> int:
        q, r = divmod(time, n - 1)
        if q % 2 == 0:
            return 1 + r
        return n - r

print(Solution().passThePillow(4, 5))   # 2
print(Solution().passThePillow(3, 2))   # 3

2
3


## 10. Vì sao thuật toán đúng

- **Đoạn đường một chiều dài `n - 1` giây:** từ người `1` đến người `n` có `n - 1` lần chuyền, từ `n` về `1` cũng `n - 1` lần. Đổi chiều **không tốn thêm giây** (gối tới đầu mút thì giây kế tiếp chuyền ngược luôn).
- **Chia `time = q * (n - 1) + r`, `0 <= r < n - 1`:** sau `q` lượt đầy đủ, gối đang ở một đầu mút; `q` chẵn → đầu mút là người `1` (lượt 0 bắt đầu ở người 1, lượt 2 ở người 1, ...), `q` lẻ → đầu mút là người `n`. Thêm `r` bước theo chiều của lượt hiện tại: tiến → `1 + r`, lùi → `n - r`. Vì `r < n - 1` nên không vượt biên (`1 + r <= n - 1`, `n - r >= 2`); lúc `r = 0` ta đang ở đúng đầu mút.
- **Cách 3 / Cách 4 (tuần hoàn):** `2 * (n - 1)` giây là một chu kỳ (đi `n - 1` + về `n - 1`), nên `time % (2 * (n - 1))` cho cùng vị trí. Trong chu kỳ, vị trí là tam giác: `r <= n - 1` thì `1 + r` (= `n - (n - 1 - r)` = `n - abs(r - (n - 1))`); `r > n - 1` thì `n - (r - (n - 1))` (= `n - abs(r - (n - 1))`). Hai nhánh gộp thành một công thức.
- **Mô phỏng (Cách 1):** invariant: sau mỗi giây `pos` đúng là người cầm gối và `step` là chiều chuyền kế tiếp (đã đảo khi ở đầu mút). Ta đảo chiều **trước khi** chuyền nên không bao giờ ra khỏi `[1, n]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng giây | `O(time)` | `O(1)` |
| 2. `divmod(time, n - 1)` | `O(1)` | `O(1)` |
| 3. Bảng một chu kỳ | `O(n)` | `O(n)` |
| 4. Công thức `abs` | `O(1)` | `O(1)` |

(Với số rất lớn, phép `%` trên số nguyên Python tốn thời gian theo số chữ số, nhưng ở đây `time <= 1000` nên coi là `O(1)`.)

## 12. Bộ test

Test 1: ví dụ đề và các edge case: `n = 2` (hàng ngắn nhất), `time` đúng bằng một chu kỳ, `time` đúng bằng lúc chạm đầu mút, và giá trị lớn nhất của đề.

In [20]:
impls = [
    ("simulate", pillow_simulate),
    ("divmod", pillow_divmod),
    ("table", pillow_table),
    ("triangle", pillow_triangle),
    ("class_divmod", SolutionDivmod().passThePillow),
    ("class_table", SolutionTable().passThePillow),
    ("class_triangle", SolutionTriangle().passThePillow),
    ("submit", Solution().passThePillow),
]
tests = [
    (4, 5, 2),
    (3, 2, 3),
    (2, 1, 2),            # hàng 2 người: 1 -> 2
    (2, 2, 1),            # rồi quay về 1
    (2, 3, 2),
    (4, 3, 4),            # chạm đầu mút n đúng lúc time = n - 1
    (4, 6, 1),            # đúng một chu kỳ 2(n-1) = 6 -> về người 1
    (5, 7, 2),            # 1,2,3,4,5,4,3,2
    (6, 13, 4),
    (1000, 1000, 1000 - 1),   # time = n: gối vừa qua người n, đang ở n - 1
    (3, 1000, 1),          # chu kỳ 4, 1000 chia hết 4 -> về người 1
]
for name, f in impls:
    for n, time, expected in tests:
        assert f(n, time) == expected, (name, n, time, f(n, time), expected)
    print(name, "PASS", len(tests), "test")

simulate PASS 11 test
divmod PASS 11 test
table PASS 11 test
triangle PASS 11 test
class_divmod PASS 11 test
class_table PASS 11 test
class_triangle PASS 11 test
submit PASS 11 test


Test 2: so khớp **mọi cặp** `(n, time)` với `n` từ 2 đến 40 và `time` từ 1 đến 1000, bằng một oracle độc lập: xây hẳn danh sách người giữ gối theo từng giây bằng cách đi qua lại một đoạn (khác cách mô phỏng `pos/step` ở trên).

In [21]:
def oracle_sequence(n, total_seconds):
    """Dãy người giữ gối ở giây 0..total_seconds, dựng bằng cách đi hết đoạn rồi quay lại."""
    seq = [1]
    forward = True
    while len(seq) <= total_seconds:
        if forward:
            seq.extend(range(2, n + 1))          # đi từ 2 đến n
        else:
            seq.extend(range(n - 1, 0, -1))      # về từ n - 1 đến 1
        forward = not forward
    return seq

for n in range(2, 41):
    seq = oracle_sequence(n, 1000)
    for time in range(1, 1001):
        expected = seq[time]
        for name, f in impls:
            assert f(n, time) == expected, (name, n, time)
print("đã so khớp n in [2, 40], time in [1, 1000]: tất cả cách đều đúng")

đã so khớp n in [2, 40], time in [1, 1000]: tất cả cách đều đúng


Test 3: ngẫu nhiên có seed cho `n` tới 1000 (đúng constraints), và tính chất: kết quả luôn trong `[1, n]`; `time` và `time + 2(n - 1)` cho cùng đáp án; `time` và `2(n - 1) - time` cho cùng đáp án (đối xứng).

In [22]:
import random
random.seed(2582)
for _ in range(2000):
    n = random.randint(2, 1000)
    time = random.randint(1, 1000)
    seq = oracle_sequence(n, time)
    expected = seq[time]
    for name, f in impls:
        assert f(n, time) == expected, (name, n, time)
    ans = Solution().passThePillow(n, time)
    assert 1 <= ans <= n
    period = 2 * (n - 1)
    if time + period <= 1000:                                        # chỉ dùng input hợp lệ
        assert Solution().passThePillow(n, time + period) == ans
    mirror = period - time % period                                  # đối xứng trong một chu kỳ
    if 1 <= mirror <= 1000:
        assert Solution().passThePillow(n, mirror) == ans
print("2000 test ngẫu nhiên + tính chất: PASS")

2000 test ngẫu nhiên + tính chất: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên đổi chiều — dùng `time % n + 1`

In [23]:
def wrong_no_bounce(n, time):
    return time % n + 1          # như thể gối đi vòng tròn 1,2,...,n,1,2,...

print(wrong_no_bounce(4, 5), "(trùng đáp án đúng 2 chỉ do ngẫu nhiên)")
print(wrong_no_bounce(3, 3), "vs đúng", Solution().passThePillow(3, 3))

2 (trùng đáp án đúng 2 chỉ do ngẫu nhiên)
1 vs đúng 2


Vì sao sai: `time % n + 1` coi hàng như **vòng tròn** (sau người `n` quay lại người 1). Đề bảo gối **đảo chiều** (sau người `n` là người `n - 1`). Ví dụ `n = 3`, `time = 3`: gối đi `1 → 2 → 3 → 2` nên là người 2, không phải người 1. Ví dụ 1 của đề "khớp" chỉ là trùng hợp — hãy luôn thử thêm test của riêng mình.

### Lỗi 2: chu kỳ sai — dùng `2 * n` thay vì `2 * (n - 1)`

In [24]:
def wrong_period(n, time):
    r = time % (2 * n)                       # SAI: chu kỳ đúng là 2 * (n - 1)
    return n - abs(r - (n - 1))

print(wrong_period(4, 7), "vs đúng", Solution().passThePillow(4, 7))

0 vs đúng 2


Vì sao sai: người 1 và người `n` mỗi người chỉ được giữ gối **một lần** trong mỗi chu kỳ, nên chu kỳ có `2n - 2` giây (không phải `2n`). Với `n = 4`: chu kỳ 6 (`1,2,3,4,3,2` rồi lại `1`), không phải 8.

### Lỗi 3: chia cho `n` thay vì `n - 1` khi dùng `divmod`

In [25]:
def wrong_divmod(n, time):
    q, r = divmod(time, n)                   # SAI: một lượt dài n - 1 giây
    return 1 + r if q % 2 == 0 else n - r

print(wrong_divmod(4, 5), "vs đúng", Solution().passThePillow(4, 5))
print(wrong_divmod(3, 3), "vs đúng", Solution().passThePillow(3, 3))

3 vs đúng 2
3 vs đúng 2


Vì sao sai: một lượt tiến từ người `1` tới người `n` chỉ gồm `n - 1` lần chuyền. Chia cho `n` làm "thương" và "dư" lệch. Hãy đếm **khoảng cách giữa các người** (`n - 1`), không đếm số người.

### Lỗi 4: đảo chiều ngay tại xuất phát (điều kiện `pos in (1, n)` đảo cả lúc đầu)

In [26]:
def wrong_start_flip(n, time):
    pos, step = 1, 1
    for _ in range(time):
        if pos == 1 or pos == n:
            step = -step                     # SAI: lúc đầu pos = 1 nên bị đảo ngay thành -1
        pos += step
    return pos

print(wrong_start_flip(4, 5), "vs đúng", Solution().passThePillow(4, 5))

-4 vs đúng 2


Vì sao sai: lúc bắt đầu gối ở người 1 và phải đi **tiến**; nếu đảo chiều vô điều kiện ở mọi đầu mút thì gối chạy ra ngoài hàng (vị trí `0`, rồi âm). Đặt chiều **trực tiếp** theo đầu mút (`pos == n` → `step = -1`; `pos == 1` → `step = 1`) thay vì `step = -step`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuyền qua lại, đụng đầu thì đổi chiều" → chuyển động **tuần hoàn** với chu kỳ `2 * (độ dài đoạn)`; độ dài đoạn là `n - 1` (số khoảng cách), không phải `n`.
- Mô phỏng với `pos, step` là cách an toàn nhất khi `time` nhỏ; hãy viết nó trước để làm oracle cho công thức.
- Muốn công thức nhanh: viết dãy 2–3 chu kỳ ra giấy (`1 2 3 4 3 2 | 1 2 3 4 3 2 ...`), thấy chu kỳ, rồi chia `% chu kỳ`.
- Mẫu "tam giác" `n - abs(r - (n - 1))` còn dùng cho bài sóng zigzag, bóng nảy, đèn chạy qua lại.
- Bài cùng ý tưởng: "Find the Child Who Has the Ball After K Seconds" (3178) — giống hệt, chỉ đánh số từ 0.

## 15. Biến thể của bài

1. Đánh số **từ 0** (người `0..n-1`) — đáp án là kết quả trừ 1.
2. Gối **dừng lại 1 giây** ở mỗi đầu mút trước khi đổi chiều (chu kỳ thành `2n`).
3. Gối xuất phát ở người `s` bất kỳ và chiều ban đầu cho trước.
4. Hỏi **giây đầu tiên** người `x` cầm gối (và lần thứ `k`): `x - 1` hoặc `2(n - 1) - (x - 1)` rồi cộng chu kỳ.
5. Hỏi **mỗi người** giữ gối bao nhiêu giây trong `time` giây đầu (đếm tần suất).
6. Hàng người **vòng tròn** (đi hết `n` thì về 1): đáp án chỉ là `time % n + 1` — chính là lỗi 1 ở mục 13 nếu áp sai bài.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | `pos`, `step`, lặp `time` lần | `O(time)` | `O(1)` | `time` nhỏ, làm oracle |
| 2. `divmod(time, n - 1)` | chẵn / lẻ lượt đi / về | `O(1)` | `O(1)` | cách nộp, dễ giải thích |
| 3. Bảng chu kỳ | `path[time % (2n - 2)]` | `O(n)` | `O(n)` | hiển thị trực quan, kiểm tra |
| 4. Công thức `abs` | `n - abs(r - (n - 1))` | `O(1)` | `O(1)` | khi cần một dòng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().passThePillow(5, 7)` bằng gì?

**Bài 2 (tính tay):** `n = 6`, `time = 13`. Viết ra dãy người giữ gối từ giây 0 đến 13 và kết luận.

**Bài 3 (điền TODO):** hoàn thành `pillow_path(n, time)` trả về **list** người giữ gối ở các giây `0, 1, ..., time` (độ dài `time + 1`).

**Bài 4 (tự viết lại):** viết `pass_pillow_triangle(n, time)` theo Cách 4 (công thức `abs`) mà không nhìn lại.

**Bài 5 (mini challenge):** viết `holding_counts(n, time)` trả về list `cnt` độ dài `n + 1` với `cnt[p]` là **số giây** (trong các mốc `0..time`) mà người `p` đang cầm gối (`cnt[0]` bỏ trống bằng `0`).

In [27]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().passThePillow(5, 7))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def pillow_path(n, time):
    path = [1]
    pos, step = 1, 1
    # TODO: lặp time lần:
    #   - nếu pos == n thì step = -1; nếu pos == 1 thì step = 1
    #   - pos += step, rồi path.append(pos)
    return path

2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** mỗi lần chuyền chỉ cộng `step`; chỉ cần đổi `step` khi gối đang ở người `1` hoặc người `n`. Chu kỳ là `2 * (n - 1)`.

**Mức 2 (vừa):** Bài 5: dựng `path = pillow_path(n, time)` rồi cộng `cnt[p] += 1` cho từng `p` trong `path`. Bài 4: `r = time % (2 * (n - 1))`, đáp án `n - abs(r - (n - 1))`.

**Mức 3 (gần đáp án):** Bài 2: `n = 6` chu kỳ `10`: `1 2 3 4 5 6 5 4 3 2 | 1 2 3 4` → giây 13 là người `4`. Bài 1: `n = 5`, chu kỳ `8`, `time = 7` → `r = 7`: `5 - abs(7 - 4) = 2`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1: n = 5, chu kỳ 8, r = 7 -> 5 - |7 - 4| = 2
assert Solution().passThePillow(5, 7) == 2

# Bài 2: 1 2 3 4 5 6 5 4 3 2 | 1 2 3 4  (giây 0..13) -> người 4
assert [pillow_table(6, t) for t in range(14)] == [1, 2, 3, 4, 5, 6, 5, 4, 3, 2, 1, 2, 3, 4]
assert Solution().passThePillow(6, 13) == 4

# Bài 3
def pillow_path_answer(n, time):
    path = [1]
    pos, step = 1, 1
    for _ in range(time):
        if pos == n:
            step = -1
        elif pos == 1:
            step = 1
        pos += step
        path.append(pos)
    return path

assert pillow_path_answer(4, 5) == [1, 2, 3, 4, 3, 2]
assert pillow_path_answer(3, 2) == [1, 2, 3]
assert pillow_path_answer(2, 3) == [1, 2, 1, 2]

# Bài 4
def pass_pillow_triangle(n, time):
    r = time % (2 * (n - 1))
    return n - abs(r - (n - 1))

for n in range(2, 12):
    for t in range(1, 60):
        assert pass_pillow_triangle(n, t) == Solution().passThePillow(n, t)

# Bài 5
def holding_counts(n, time):
    cnt = [0] * (n + 1)
    for p in pillow_path_answer(n, time):
        cnt[p] += 1
    return cnt

assert holding_counts(4, 5) == [0, 1, 2, 2, 1]
assert sum(holding_counts(7, 30)) == 31
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template A — mô phỏng với chiều:**

```python
pos, step = 1, 1
for _ in range(time):
    if pos == n:   step = -1
    elif pos == 1: step = 1
    pos += step
return pos
```

**Template B — công thức (chu kỳ `2 * (n - 1)`):**

```python
q, r = divmod(time, n - 1)
return 1 + r if q % 2 == 0 else n - r        # hoặc: n - abs(time % (2*(n-1)) - (n-1))
```

**Checklist trước khi submit:**

- [ ] Chu kỳ là `2 * (n - 1)`, không phải `2 * n`?
- [ ] `divmod(time, n - 1)` (chia cho **khoảng cách** `n - 1`)?
- [ ] `q` chẵn → `1 + r`, `q` lẻ → `n - r`?
- [ ] Không dùng `time % n + 1` (đó là hàng **vòng tròn**)?
- [ ] Nếu mô phỏng: đặt `step` theo đầu mút, không đảo chiều ngay lúc xuất phát?
- [ ] Đã thử `n = 2`, `time` bằng đúng một chu kỳ, `time = n - 1`?
- [ ] Chữ ký đúng `passThePillow(self, n: int, time: int) -> int`?